# 예제 하드웨어 및 핀 연결 정보

- Husky
- CW308 (with STM32F F303)

| CW308 타겟 보드 (JTAG/SWD 헤더) | ChipWhisperer-Husky (USERIO 핀) |
|--------------------------------|--------------------------------|
| TMS (또는 SWDIO)              | D0                             |
| TCK (또는 SWCLK)              | D1                             |
| TDO (또는 SWO)                | D2                             |

# (실천) 칩위스퍼러 플랫폼 설정 & 사전 작업 & 바이너리 컴파일 및 프로그램

In [2]:
#PLATFORM = 'CWHUSKY'
PLATFORM = 'CW308_STM32F3'

In [3]:
%run '../base/My_Setup.ipynb'

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
Welcome to another exciting ChipWhisperer target build!!
.
Cleaning project:
rm -f -- simpleserial-base-CW308_CC2538.hex simpleserial-base-CW301_AVR.hex simpleserial-base-CW303.hex simpleserial-base-CW304.hex simpleserial-base-CW308_MEGARF.hex simpleserial-base-CW308_SAM4L.hex simpleserial-base-CW308_STM32F0.hex simpleserial-base-CW308_STM32F1.hex simpleserial-base-CW308_STM32F2.hex simpleserial-base-CW308_STM32F3.hex simpleserial-base-CW308_STM32F4.hex simpleserial-base-CW308_K24F.hex simpleserial-base-CW308_NRF52.hex simpleserial-base-CW308_AURIX.hex simpleserial-base-CW308_SAML11.hex simpleserial-base-CW308_EFM32TG11B.hex simpleserial-base-CWLITEARM.hex simpleserial-base-CWLITEXMEGA.hex simpleserial-base-CWNANO.hex simpleserial-base-CWHUSKY.hex simpleserial-base-CW308_K82F.hex simpleserial-base-CW308_PSOC62.hex simpleserial-base-CW308_IMXRT1062.hex simpleserial-base-CW308_FE310.hex simpleserial-base-CW308_EFR32MG21A.hex simpleseria

OSError: Could not find ChipWhisperer. Is it connected?

OSError: Could not find ChipWhisperer. Is it connected?

OSError: Could not find ChipWhisperer. Is it connected?

- 스코프 초기화 및 타겟보드 리셋

In [13]:
scope.default_setup()

# 4 샘플 per 클럭
scope.clock.adc_src = 'clkgen_x4'
scope.clock.clkgen_src = 'system'
scope.clock.adc_mul = 4
scope.adc.decimate = 1

# 트레이스 모듈 활성화 및 샘플 수 설정
scope.trace.target = target
scope.trace.enabled = True

# 타겟보드 리셋 함수 reset_target(scope)  ->  ../base/Setup_Generic.ipynb  
reset_target(scope)
time.sleep(1)
reset_target(scope)
time.sleep(1)

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:1628) scope.clock.adc_src is provided for backwards compability, but scope.clock.clkgen_src and scope.clock.adc_mul should be used for Husky.


scope.adc.samples                        changed from 948                       to 5000                     


- 사전 작업

In [14]:
MAX_DATA_LEN = 16 

# 사전 샘플링 개수
NUM_PRESAMPLES = 10


# init
my_fsr_cmd(target, 0x81, 'k', bytearray(MAX_DATA_LEN))
my_fsr_cmd(target, 0x81, 'p', bytearray(MAX_DATA_LEN))
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# 샘플 개수 설정
MAX_TR_LEN = my_setting_num_samples(target, scope, NUM_PRESAMPLES)  

In [1]:
scope.trace.clock.fe_clock_src = 'target_clock'
assert scope.trace.clock.fe_clock_alive, "선택한 클럭이 활성화되지 않았습니다."

scope.trace.trace_mode = 'SWO'
scope.trace.jtag_to_swd() # 타겟을 SWO 모드로 스위칭

acpr = 0
trigger_freq_mul = 8
scope.trace.clock.swo_clock_freq = scope.clock.clkgen_freq * trigger_freq_mul
scope.trace.target_registers.TPI_ACPR = acpr
scope.trace.swo_div = trigger_freq_mul * (acpr + 1)

assert scope.trace.clock.swo_clock_locked, "트리거/UART 클럭이 고정되지 않았습니다."
assert scope.userio.pins[2].status, "SWO 라인이 High 상태가 아닙니다. 점퍼선 연결(TDO -> D2)을 확인하세요!"

NameError: name 'scope' is not defined

- 다수의 파형 수집

In [ ]:
debug_mode = False   #True False

NUM_OF_TRACES = 50 
FileDir = '../traces'
TV_case = 'random'    # fixed  random 


if debug_mode :
    NUM_OF_TRACES = 5    
    array_i_k = []
    array_i_p = []
    array_o = []
    array_t = []

with h5py.File(f'{FileDir}/tmp_SCA_DB.h5', 'w') as f: 
    dset_i_k = f.create_dataset('i_k', shape=(0, MAX_DATA_LEN), \
                              maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)
    dset_i_p = f.create_dataset('i_p', shape=(0, MAX_DATA_LEN), \
                              maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)


    dset_o = f.create_dataset('o', shape=(0, MAX_DATA_LEN), \
                              maxshape=(None, MAX_DATA_LEN), dtype='uint8', chunks=True)

    dset_t = f.create_dataset('t', shape=(0, MAX_TR_LEN), \
                              maxshape=(None, MAX_TR_LEN), dtype='float32', chunks=True)

    
    for i in trange(NUM_OF_TRACES, desc='Capturing traces'):
    
        if TV_case == 'fixed':
            data_i_k = bytearray(b'\xA0' * (MAX_DATA_LEN)) 
            data_i_p = bytearray(b'\x08' * (MAX_DATA_LEN))       
        elif TV_case == 'random':
            data_i_k = bytearray([random.randint(0,255) for _ in range(MAX_DATA_LEN)])  
            data_i_p = bytearray([random.randint(0,255) for _ in range(MAX_DATA_LEN)])       
        else :
            print('TV_case is invalid')
            break
        
        # init
        if (
            data_i_k != my_fsr_cmd(target, 0x81, 'k', data_i_k, payload_only=True) or
            data_i_p != my_fsr_cmd(target, 0x81, 'p', data_i_p, payload_only=True) or
            MAX_DATA_LEN != my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]), payload_only=True)[0]
        ):
            print('init fail!')
            continue

        # 파형 수집
        try:
            data_o, trace = my_get_trace(target, scope)
        except (TimeoutError, ValueError) as e:
            print(e) 
            continue                   
                
        if debug_mode :         
            print(f'---- for {i} ----')      
            print(f'input  : {data_i_k.hex(" ")} \n')   
            print(f'input  : {data_i_p.hex(" ")} \n')          
            print(f'output : {data_o.hex(" ")} \n')

            array_i_k.append(data_i_k)
            array_i_p.append(data_i_p)
            array_o.append(data_o)  
            array_t.append(trace)
            
        else :
            dset_i_k.resize(dset_i_k.shape[0] + 1, axis=0)
            dset_i_p.resize(dset_i_p.shape[0] + 1, axis=0)
            dset_o.resize(dset_o.shape[0] + 1, axis=0)
            dset_t.resize(dset_t.shape[0] + 1, axis=0)
            
            dset_i_k[-1:] = np.array(data_i_k)
            dset_i_p[-1:] = np.array(data_i_p)
            dset_o[-1:] = np.array(data_o)
            dset_t[-1:] = np.array(trace)
    
    
if debug_mode :        
    %matplotlib inline    
    fig = plt.plot(np.transpose(np.array(array_t)))  
else :
    d = time.strftime('%Y%m%d-%H%M%S', time.localtime(time.time()))
    file = Path(f'{FileDir}/tmp_SCA_DB.h5')
    file.rename(f'{FileDir}/{d}_SCA_DB.h5')

print('Done!\n\n')

In [ ]:
scope.dis()
target.dis()

In [ ]:
# 파일 경로 설정
FileDir = '../traces'
file_path = Path(FileDir) / '20260422-102045_SCA_DB.h5'   # 파일명 수정

# 데이터 로드
with h5py.File(file_path, 'r') as f:
    i_k = f['i_k'][:]   # key input
    i_p = f['i_p'][:]   # plaintext
    o   = f['o'][:]     # output
    t   = f['t'][:]     # trace

# 확인
print('i_k shape:', i_k.shape)
print('i_p shape:', i_p.shape)
print('o shape  :', o.shape)
print('t shape  :', t.shape)


idx = 0   # 확인할 인덱스
print(f'\n[샘플 #{idx}]')
print(f'  i_k : {bytes(i_k[idx]).hex(" ")}')
print(f'  i_p : {bytes(i_p[idx]).hex(" ")}')
print(f'  o   : {bytes(o[idx]).hex(" ")}')

# 샘플 파형 (5개) 확인
plt.plot(t[:5].T)       # Transpose 주의
plt.title("Sample Traces")
plt.show()